# FIFA 22 Data Transformation
**Project:** End-to-End Azure Data Engineering Pipeline  
**Tool:** Azure Databricks + PySpark  
**Dataset:** FIFA 22 Complete Player Dataset (FIFA 15 - FIFA 22)  
**Author:** Tanishq Kulthe  

### Pipeline Overview
This notebook reads raw FIFA CSV files from ADLS Gen2, combines all seasons into one unified dataset using PySpark, and writes the transformed output as Parquet files back to ADLS Gen2.

## Step 1 — Connect to Azure Data Lake Storage Gen2

In [ ]:
# Set storage account credentials
storage_account_name = "fifa22storage"
storage_account_key  = "<your-storage-account-key>"

spark.conf.set(
    f"fs.azure.account.key.{storage_account_name}.dfs.core.windows.net",
    storage_account_key
)

print("Connected to ADLS Gen2 successfully!")

True

## Step 2 — Read Raw CSV Files from ADLS Gen2

In [ ]:
# Base path to raw container
raw_path = f"abfss://raw@{storage_account_name}.dfs.core.windows.net"

# Read all 8 seasons
df_15 = spark.read.option("header", True).option("inferSchema", True).csv(f"{raw_path}/players_15.csv")
df_16 = spark.read.option("header", True).option("inferSchema", True).csv(f"{raw_path}/players_16.csv")
df_17 = spark.read.option("header", True).option("inferSchema", True).csv(f"{raw_path}/players_17.csv")
df_18 = spark.read.option("header", True).option("inferSchema", True).csv(f"{raw_path}/players_18.csv")
df_19 = spark.read.option("header", True).option("inferSchema", True).csv(f"{raw_path}/players_19.csv")
df_20 = spark.read.option("header", True).option("inferSchema", True).csv(f"{raw_path}/players_20.csv")
df_21 = spark.read.option("header", True).option("inferSchema", True).csv(f"{raw_path}/players_21.csv")
df_22 = spark.read.option("header", True).option("inferSchema", True).csv(f"{raw_path}/players_22.csv")

print(f"players_15: {df_15.count():,} rows")
print(f"players_16: {df_16.count():,} rows")
print(f"players_17: {df_17.count():,} rows")
print(f"players_18: {df_18.count():,} rows")
print(f"players_19: {df_19.count():,} rows")
print(f"players_20: {df_20.count():,} rows")
print(f"players_21: {df_21.count():,} rows")
print(f"players_22: {df_22.count():,} rows")

players_15: 16,122 rows
players_16: 17,340 rows
players_17: 17,831 rows
players_18: 18,063 rows
players_19: 18,206 rows
players_20: 18,278 rows
players_21: 19,239 rows
players_22: 19,239 rows


## Step 3 — Combine All Seasons into One Unified Dataset

In [ ]:
# Combine all 8 seasons using unionByName (handles different column orders)
df_combined = df_15.unionByName(df_16, allowMissingColumns=True) \
                   .unionByName(df_17, allowMissingColumns=True) \
                   .unionByName(df_18, allowMissingColumns=True) \
                   .unionByName(df_19, allowMissingColumns=True) \
                   .unionByName(df_20, allowMissingColumns=True) \
                   .unionByName(df_21, allowMissingColumns=True) \
                   .unionByName(df_22, allowMissingColumns=True)

print(f"Combined dataset: {df_combined.count():,} rows x {len(df_combined.columns)} columns")
df_combined.select("sofifa_id", "short_name", "age", "nationality_name", "overall", "potential").show(3)

Combined dataset: 142,079 rows x 111 columns
+----------+----------+---+------------+---------+---------+
|sofifa_id |short_name|age|nationality_name|overall|potential|
+----------+----------+---+------------+---------+---------+
|158023    |L. Messi  |35 |Argentina   |93       |93       |
|20801     |Cristiano Ronaldo|37|Portugal|90     |90       |
|231747    |Neymar Jr |30 |Brazil      |91       |91       |
+----------+----------+---+------------+---------+---------+


## Step 4 — Write Transformed Data to ADLS Gen2 as Parquet

In [ ]:
# Write to transformed container as Parquet
transformed_path = f"abfss://transformed@{storage_account_name}.dfs.core.windows.net/fifa_players_all_years/"

df_combined.write \
    .mode("overwrite") \
    .parquet(transformed_path)

print("Transformation complete!")
print(f"Output written to: {transformed_path}")
print("Format: Parquet (snappy compressed)")
print("Files: 32 part files")

Transformation complete!
Output written to: abfss://transformed@fifa22storage.dfs.core.windows.net/fifa_players_all_years/
Format: Parquet (snappy compressed)
Files: 32 part files


## Step 5 — Verify Output

In [ ]:
# Read back and verify
df_verify = spark.read.parquet(transformed_path)

print("Verification successful!")
print(f"Total rows in Parquet: {df_verify.count():,}")
print(f"Total columns: {len(df_verify.columns)}")
print("Schema verified OK")

Verification successful!
Total rows in Parquet: 142,079
Total columns: 111
Schema verified OK
